# Note:

This file was used for initial exploration and is saved for reference, the logic here has been formalised in the `compare_summaries.py` file. 

In [ ]:
from pathlib import Path
from pydantic.v1 import BaseModel, Field

from langchain.chat_models import ChatOpenAI
from langchain.agents import Tool
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.document_loaders import TextLoader
from langchain.chains import RetrievalQA
from langchain.agents import initialize_agent
from langchain.agents import AgentType
from langchain.llms import OpenAI
from langchain import PromptTemplate
from langchain.prompts.chat import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
)

# Plan

1. Use document compare against best summaries
2. Try prompt engineering for comparison

The sections summary seem to do the best job at capturing the details of the contracts so I will use them. 

In [ ]:
save_dir = Path("data", "summary_comparison")
summaries_dir = Path("data", "summaries")
sections_2015_fname = Path(summaries_dir, "2015_sections_summary.txt")
sections_2023_fname = Path(summaries_dir, "2023_sections_summary.txt")

# Document Comparison Tool

In [ ]:
class DocumentInput(BaseModel):
    question: str = Field()


llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo-0613")

tools = []
files = [
    {
        "name": "2015-TandCS",
        "path": str(sections_2015_fname),
    },
    {
        "name": "2023-TandCS",
        "path": str(sections_2023_fname),
    },
]

for file in files:
    loader = TextLoader(file["path"])
    pages = loader.load_and_split()
    text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
    docs = text_splitter.split_documents(pages)
    embeddings = OpenAIEmbeddings()
    retriever = FAISS.from_documents(docs, embeddings).as_retriever()

    # Wrap retrievers in a Tool
    tools.append(
        Tool(
            args_schema=DocumentInput,
            name=file["name"],
            description=f"useful when you want to answer questions about {file['name']}",
            func=RetrievalQA.from_chain_type(llm=llm, retriever=retriever),
        )
    )

In [ ]:
agent = initialize_agent(
    agent=AgentType.OPENAI_FUNCTIONS,
    tools=tools,
    llm=llm,
    verbose=True,
)

In [ ]:
query = "What are the key differences between the 2015-TandCS and 2023-TandCS?"
output = agent({"input": query})

In [ ]:
with open(Path(save_dir, "document_comparison_output.txt"), "w") as f:
    f.write(output["output"])

# Prompt Engineering - Davinci

In [ ]:
openai = OpenAI(model="text-davinci-003", temperature=0)

In [ ]:
with open(sections_2015_fname, "r") as f:
    section_2015 = f.read()
with open(sections_2023_fname, "r") as f:
    section_2023 = f.read()

In [ ]:
prompt = """Compare the two texts provided below and provide the key differences between them. 
The output should be aimed towards legal experts. 
The first text includes information from 2015 and is contained with three backticks (```). 
The second text contains information from 2023 and is contained wihin three hash symbols (###).
Your answer should be detailed but specific, formatted as bullet points and refer back to the text names when describing the differences. 
                   
                      
2015 text: ```{text1}```
            
2023 text: ###{text2}###
"""

prompt_template = PromptTemplate(input_variables=["text1", "text2"], template=prompt)

In [ ]:
prompt_template_formatted = prompt_template.format(
    text1=section_2015, text2=section_2023
)

In [ ]:
prompt_engineering_output = openai(prompt_template_formatted)

In [ ]:
print(prompt_engineering_output)

In [ ]:
with open(Path(save_dir, "prompt_engineering_output_davinci.txt"), "w") as f:
    f.write(prompt_engineering_output)

# Prompt Engineering - GPT

In [ ]:
human_message_prompt = HumanMessagePromptTemplate.from_template(prompt)

chat_prompt = ChatPromptTemplate.from_messages([human_message_prompt])

In [ ]:
gpt_prompt_engineering_output = llm(
    chat_prompt.format_prompt(text1=section_2015, text2=section_2023).to_messages()
)

In [ ]:
print(gpt_prompt_engineering_output.content)

In [ ]:
with open(Path(save_dir, "prompt_engineering_output_gpt.txt"), "w") as f:
    f.write(gpt_prompt_engineering_output.content)